# 04 — Entraînement des Modèles (sur France uniquement)

Compare 5 algorithmes (Random Forest, Gradient Boosting, XGBoost, Régression
Logistique, SVM) via GridSearchCV avec validation croisée stratifiée 5-fold.

🔒 Entraînement exclusivement sur France. Cameroun n'est JAMAIS vu ici.


In [1]:
import sys
from pathlib import Path
# Ajout robuste de src/ au path, peu importe le dossier depuis lequel le kernel démarre
project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.append(str(project_root / 'src'))
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from model_utils import get_model_zoo, train_with_gridsearch, compute_clinical_metrics

X_france_A = np.load('../data/processed/X_france_A.npy')
X_france_B = np.load('../data/processed/X_france_B.npy')
y_france = pd.read_csv('../data/processed/y_france.csv').iloc[:,0]

print("X_france_A:", X_france_A.shape, "| y_france positifs:", y_france.sum())


X_france_A: (30000, 22) | y_france positifs: 3377


## 4.1 Split interne train/test (pour sélection de modèle, PAS pour validation finale)

In [2]:
X_train_A, X_test_A, y_train, y_test = train_test_split(
    X_france_A, y_france, test_size=0.2, stratify=y_france, random_state=42)

X_train_B, X_test_B, _, _ = train_test_split(
    X_france_B, y_france, test_size=0.2, stratify=y_france, random_state=42)

print(f"Train: {X_train_A.shape[0]} | Test interne: {X_test_A.shape[0]}")
print("Note: ce test interne sert UNIQUEMENT à comparer les 5 algos entre eux.")
print("La vraie validation se fait au notebook 05, sur Cameroun.")


Train: 24000 | Test interne: 6000
Note: ce test interne sert UNIQUEMENT à comparer les 5 algos entre eux.
La vraie validation se fait au notebook 05, sur Cameroun.


## 4.2 Entraînement Modèle A — comparaison des 5 algorithmes

In [3]:
from sklearn.utils.class_weight import compute_sample_weight

model_zoo = get_model_zoo()
results_A = {}
fitted_models_A = {}

# GradientBoostingClassifier n'a PAS de paramètre class_weight natif (contrairement
# à RandomForest/LogReg/SVM). On lui applique donc un sample_weight équivalent
# pour une comparaison équitable entre les 5 algorithmes.
sample_weight_balanced = compute_sample_weight(class_weight='balanced', y=y_train)

for name, cfg in model_zoo.items():
    print(f"\n{'='*50}\nEntraînement: {name}")
    sw = sample_weight_balanced if name == 'GradientBoosting' else None
    best_model, best_params, best_cv_score = train_with_gridsearch(
        cfg['estimator'], cfg['param_grid'], X_train_A, y_train, sample_weight=sw)
    y_pred = best_model.predict(X_test_A)
    y_proba = best_model.predict_proba(X_test_A)[:,1]
    metrics = compute_clinical_metrics(y_test, y_pred, y_proba)
    results_A[name] = metrics
    fitted_models_A[name] = best_model
    print(f"Best params: {best_params}")
    print(f"Test interne -> Sensibilité: {metrics['sensibilite']:.3f} | AUC: {metrics['auc_roc']:.3f}")



Entraînement: RandomForest
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Best params: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 200}
Test interne -> Sensibilité: 0.053 | AUC: 0.673

Entraînement: GradientBoosting
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 400}
Test interne -> Sensibilité: 0.524 | AUC: 0.709

Entraînement: XGBoost
Fitting 5 folds for each of 24 candidates, totalling 120 fits


c:\PROJECTS\GDM_VALIDATION_PROJECT\.venv\Lib\site-packages\xgboost\core.py:158: UserWarning: [20:18:28] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-06abd128ca6c1688d-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Best params: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 400, 'scale_pos_weight': 1}
Test interne -> Sensibilité: 0.086 | AUC: 0.696

Entraînement: LogisticRegression
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best params: {'C': 1, 'penalty': 'l2'}
Test interne -> Sensibilité: 0.683 | AUC: 0.751

Entraînement: SVM
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best params: {'C': 10, 'kernel': 'rbf'}
Test interne -> Sensibilité: 0.599 | AUC: 0.696


In [4]:
results_A_df = pd.DataFrame(results_A).T
results_A_df = results_A_df.sort_values('auc_roc', ascending=False)
print("=== Comparaison des 5 modèles (Modèle A) — Test interne France ===")
results_A_df[['sensibilite','specificite','accuracy','auc_roc','f1_score']]


=== Comparaison des 5 modèles (Modèle A) — Test interne France ===


,sensibilite,specificite,accuracy,auc_roc,f1_score
LogisticRegression,0.683,0.6945,0.6932,0.7509,0.3337
GradientBoosting,0.5244,0.7927,0.7625,0.7091,0.3319
SVM,0.5985,0.7157,0.7025,0.6961,0.3116
XGBoost,0.0859,0.9822,0.8813,0.696,0.1401
RandomForest,0.0533,0.9816,0.8772,0.6727,0.089


## 4.3 Sélection du meilleur modèle (Modèle A)

In [5]:
best_model_name_A = results_A_df['auc_roc'].idxmax()
best_model_A = fitted_models_A[best_model_name_A]
print(f"🏆 Meilleur modèle (Modèle A): {best_model_name_A}")
print(results_A_df.loc[best_model_name_A])


🏆 Meilleur modèle (Modèle A): LogisticRegression
n_total                         6000
n_positifs                       675
n_negatifs                      5325
TP                               461
TN                              3698
FP                              1627
FN                               214
sensibilite                    0.683
specificite                   0.6945
accuracy                      0.6932
precision_PPV                 0.2208
npv                           0.9453
f1_score                      0.3337
auc_roc                       0.7509
kpi_sensibilite_85_atteint     False
kpi_specificite_75_atteint     False
kpi_auc_085_atteint            False
Name: LogisticRegression, dtype: object


## 4.4 Entraînement Modèle B (ablation — + niveau_etude) avec le même algo gagnant

In [6]:
from sklearn.base import clone
model_zoo_B = get_model_zoo()
best_model_B, best_params_B, _ = train_with_gridsearch(
    model_zoo_B[best_model_name_A]['estimator'],
    model_zoo_B[best_model_name_A]['param_grid'],
    X_train_B, y_train)

y_pred_B = best_model_B.predict(X_test_B)
y_proba_B = best_model_B.predict_proba(X_test_B)[:,1]
metrics_B = compute_clinical_metrics(y_test, y_pred_B, y_proba_B)

print(f"Modèle A ({best_model_name_A}) — AUC test interne: {results_A_df.loc[best_model_name_A,'auc_roc']:.4f}")
print(f"Modèle B ({best_model_name_A}+niveau_etude) — AUC test interne: {metrics_B['auc_roc']:.4f}")
print(f"\nGain AUC en ajoutant niveau_etude: {metrics_B['auc_roc'] - results_A_df.loc[best_model_name_A,'auc_roc']:+.4f}")
print("-> Si gain < 0.01-0.02, confirme l'hypothèse: niveau_etude = bruit, pas de valeur ajoutée")


Fitting 5 folds for each of 4 candidates, totalling 20 fits
Modèle A (LogisticRegression) — AUC test interne: 0.7509
Modèle B (LogisticRegression+niveau_etude) — AUC test interne: 0.7517

Gain AUC en ajoutant niveau_etude: +0.0008
-> Si gain < 0.01-0.02, confirme l'hypothèse: niveau_etude = bruit, pas de valeur ajoutée


## 4.5 Sauvegarde des modèles finaux

In [7]:
joblib.dump(best_model_A, '../models/model_A.pkl')
joblib.dump(best_model_B, '../models/model_B.pkl')
results_A_df.to_csv('../results/comparaison_5_modeles_A.csv')

with open('../results/best_model_info.txt', 'w') as f:
    f.write(f"Meilleur modèle: {best_model_name_A}\n")
    f.write(f"Params: {best_params_B}\n")
    f.write(f"AUC Modèle A (test interne France): {results_A_df.loc[best_model_name_A,'auc_roc']:.4f}\n")
    f.write(f"AUC Modèle B (test interne France): {metrics_B['auc_roc']:.4f}\n")

print("✅ Modèles sauvegardés dans models/")
print("⚠️ RAPPEL: ces métriques sont sur test interne FRANCE, pas encore la validation externe Cameroun.")


✅ Modèles sauvegardés dans models/
⚠️ RAPPEL: ces métriques sont sur test interne FRANCE, pas encore la validation externe Cameroun.


➡️ **Suite : notebook 05_external_validation.ipynb — LE notebook clé de la thèse**